# 14 — Voice Assistant Pipeline

## Objective

In this notebook, we will build a simple end-to-end voice assistant pipeline by combining speech recognition, language understanding, and text-to-speech generation.

The pipeline will follow:

User Speech
→ ASR
→ Text
→ Language Model
→ Response
→ TTS
→ Generated Speech

We will also measure the latency of the main processing stages.

## Pipeline overview

The voice assistant is composed of three main AI components:

1. Automatic Speech Recognition (ASR)
2. Language Model (LLM)
3. Text-to-Speech (TTS)

Each component performs a different task, but together they create a complete conversational voice system.

## Environment setup

We will first import the main libraries and check the available hardware.

The assistant will combine:

- Whisper for speech recognition;
- a language model for response generation;
- a TTS model for speech synthesis;
- latency measurements for each stage.

In [ ]:
import time
import torch
import librosa
import numpy as np

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Device: CPU")

## Install dependencies

We will install the libraries required for the three main components of the voice assistant:

- Whisper for speech recognition;
- a lightweight language model for response generation;
- a text-to-speech system for voice synthesis.

In [ ]:
!pip install -q transformers accelerate librosa soundfile sentencepiece

## Step 1 — Speech Recognition

The first component of the voice assistant converts the user's speech into text.

We will use Whisper as the Automatic Speech Recognition (ASR) model and measure the transcription latency.

In [ ]:
from transformers import pipeline
import time

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0
)

print("Whisper loaded successfully")

## Transcribe the user audio

We will now send an audio sample to Whisper and measure how long the transcription takes.

This gives us the latency of the ASR stage in the voice assistant pipeline.

In [ ]:
audio_path = "/content/sunny-day-weather-forecast.mp3"

start_time = time.perf_counter()

result = asr(audio_path, return_timestamps=True)

asr_latency = time.perf_counter() - start_time

transcript = result["text"]

print("Transcript:")
print(transcript)

print("\nASR latency:", round(asr_latency, 2), "seconds")

### ASR result

Whisper successfully transcribed the weather forecast.

The model extracted detailed information such as:

- morning temperature: 58°F;
- noon temperature: 75°F;
- evening temperature: 65°F;
- moderate UV index;
- sunset around 7:45 PM;
- recommended outdoor activities.

The ASR latency was approximately 11.91 seconds for the full audio file.

This confirms that Whisper can reliably convert a spoken weather report into text that can later be processed by a language model.

## Step 2 — Language Model

The second component of the voice assistant processes the transcript and generates an answer to the user's question.

Instead of answering from general knowledge, the model will use the weather transcript as context.

Pipeline:

Weather Audio
→ Whisper
→ Transcript
→ Language Model
→ Answer

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=0,
    torch_dtype=torch.float16
)

print("Language model loaded successfully")

## Ask a question about the weather

The language model will now use the Whisper transcript as context and answer a question about the weather forecast.

This simulates the reasoning stage of the voice assistant.

In [ ]:
question = "What will the temperature be around noon?"

prompt = f"""
You are a weather assistant.

Use only the weather forecast below to answer the question.

Weather forecast:
{transcript}

Question:
{question}

Answer briefly and clearly.
"""

start_time = time.perf_counter()

response = llm(
    prompt,
    max_new_tokens=80,
    do_sample=False
)

llm_latency = time.perf_counter() - start_time

answer = response[0]["generated_text"][len(prompt):].strip()

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nLLM latency:", round(llm_latency, 2), "seconds")

### LLM result

The language model correctly answered the weather question using the Whisper transcript as context.

For the question:

"What will the temperature be around noon?"

the model correctly identified a temperature of 75°F.

The LLM stage took approximately 6.02 seconds.

This demonstrates how the transcript generated by the ASR model can be used as contextual information for a conversational weather assistant.

## Step 3 — Text-to-Speech

The final AI component converts the language model's text response into speech.

This allows the voice assistant to respond to the user using generated audio.

Pipeline:

LLM Answer
→ Text-to-Speech
→ Spoken Response

In [ ]:
!pip install -q transformers accelerate soundfile

In [ ]:
from transformers import pipeline

tts = pipeline(
    "text-to-speech",
    model="facebook/mms-tts-eng",
    device=0
)

print("TTS model loaded successfully")

## Generate the spoken response

The text answer generated by the language model will now be converted into speech.

We will also measure the latency of the Text-to-Speech stage.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf
import time

start_time = time.perf_counter()

tts_output = tts(answer)

tts_latency = time.perf_counter() - start_time

audio_array = tts_output["audio"]
sampling_rate = tts_output["sampling_rate"]

output_path = "/content/weather_assistant_response.wav"

sf.write(
    output_path,
    audio_array.squeeze(),
    sampling_rate
)

print("TTS latency:", round(tts_latency, 2), "seconds")

display(Audio(audio_array, rate=sampling_rate))

### TTS result

The Text-to-Speech model successfully converted the language model's response into spoken audio.

The generated speech was clear and understandable.

The TTS stage took approximately 1.70 seconds.

## End-to-End Latency

A complete voice assistant combines the latency of multiple components.

We will calculate the total processing time across:

- Automatic Speech Recognition;
- Language Model inference;
- Text-to-Speech generation.

In [ ]:
total_latency = asr_latency + llm_latency + tts_latency

print("ASR latency:", round(asr_latency, 2), "seconds")
print("LLM latency:", round(llm_latency, 2), "seconds")
print("TTS latency:", round(tts_latency, 2), "seconds")

print("\nTotal pipeline latency:", round(total_latency, 2), "seconds")

### End-to-end latency analysis

The complete voice assistant pipeline took approximately 19.63 seconds.

Latency breakdown:

- ASR: 11.91 seconds
- LLM: 6.02 seconds
- TTS: 1.70 seconds

The ASR stage was the slowest component because the input was a full weather forecast rather than a short user question.

For a real voice assistant, shorter audio inputs and streaming processing would significantly reduce the perceived latency.

## Build the complete voice assistant function

We will now combine the ASR, language model, and TTS components into a single function.

The function will:

1. transcribe the input audio;
2. answer a weather question using the transcript as context;
3. generate the spoken response;
4. return the latency of each stage.

In [ ]:
def voice_assistant(audio_path, question):
    # Step 1 — ASR
    start = time.perf_counter()
    asr_result = asr(audio_path, return_timestamps=True)
    transcript = asr_result["text"]
    asr_time = time.perf_counter() - start

    # Step 2 — LLM
    prompt = f"""
    You are a weather assistant.

    Use only the weather forecast below to answer the question.

    Weather forecast:
    {transcript}

    Question:
    {question}

    Answer briefly and clearly.
    """

    start = time.perf_counter()

    response = llm(
        prompt,
        max_new_tokens=80,
        do_sample=False
    )

    llm_time = time.perf_counter() - start

    answer = response[0]["generated_text"][len(prompt):].strip()

    # Step 3 — TTS
    start = time.perf_counter()
    tts_output = tts(answer)
    tts_time = time.perf_counter() - start

    audio_array = tts_output["audio"]
    sampling_rate = tts_output["sampling_rate"]

    total_time = asr_time + llm_time + tts_time

    return {
        "transcript": transcript,
        "answer": answer,
        "audio": audio_array,
        "sampling_rate": sampling_rate,
        "asr_latency": asr_time,
        "llm_latency": llm_time,
        "tts_latency": tts_time,
        "total_latency": total_time
    }

## Test the complete voice assistant

We will now run the full pipeline using a weather question.

The assistant should:

- transcribe the weather forecast;
- answer the question using the transcript;
- generate a spoken response;
- report the latency of each stage.

In [ ]:
question = "What is the UV index today?"

result = voice_assistant(
    "/content/sunny-day-weather-forecast.mp3",
    question
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

print("\nLatency:")
print("ASR:", round(result["asr_latency"], 2), "s")
print("LLM:", round(result["llm_latency"], 2), "s")
print("TTS:", round(result["tts_latency"], 2), "s")
print("Total:", round(result["total_latency"], 2), "s")

display(
    Audio(
        result["audio"],
        rate=result["sampling_rate"]
    )
)

### Complete pipeline result

The complete weather voice assistant successfully processed the forecast and answered a new question.

For the question:

"What is the UV index today?"

the assistant correctly identified the UV index as moderate and generated a spoken response.

### Latency

- ASR: 9.97 seconds
- LLM: 2.73 seconds
- TTS: 0.81 seconds
- Total: 13.51 seconds

The ASR stage remains the main source of latency because the complete weather forecast is transcribed before each question.

In a production system, the forecast could be transcribed once and reused for multiple questions, significantly reducing the response time.

## Conclusion

In this notebook, we built a complete voice assistant pipeline for weather-related question answering.

### What we implemented

- speech recognition using Whisper;
- weather transcript extraction;
- contextual question answering using a language model;
- Text-to-Speech response generation;
- latency measurement for each component;
- an end-to-end voice assistant function.

### Final pipeline

User / Weather Audio
→ Whisper ASR
→ Transcript
→ Language Model
→ Answer
→ Text-to-Speech
→ Spoken Response

### Key takeaway

A voice assistant is not a single model but a pipeline of specialized AI components.

The main challenge is not only generating correct answers, but also coordinating ASR, language understanding, and speech synthesis while keeping the overall latency low.